In [5]:
!pip install "numpy<2.0.0"
# Then restart terminal

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 121.7 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.0
    Uninstalling numpy-2.2.0:
      Successfully uninstalled numpy-2.2.0


In [1]:
!pip install -qq medmnist
!pip install -qq av
!pip install opencv-python

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import sys
sys.path.append('/home/ubuntu/.lambda/lib/python3.10/site-packages')

In [3]:
import os
import cv2
import numpy as np
import tensorflow as tf  # for data preprocessing only
import keras
import av
import csv
import math
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

2024-12-17 21:48:53.942273: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-12-17 21:48:53.954486: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-12-17 21:48:53.969094: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-12-17 21:48:53.973439: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-12-17 21:48:53.983742: I tensorflow/core/platform/cpu_feature_guar

In [4]:
DATA_DIR = 'dataset2/videos'
LABEL_DIRS = ['dataset2/annotations1', 'dataset2/annotations2', 'dataset2/annotations3', 
              'dataset2/annotations4', 'dataset2/annotations5']
NUM_CLASSES = 7

TEST_SIZE = 0.2
VAL_SIZE = 0.2
DATASET_SEED = 81

FRAME_SIZE = (360, 240)
SCALE = 255.0
CLIP_LENGTH = 60 # 120 runs into errors

INPUT_SHAPE = (210, 270, 1) # coordinates are inverted for some reason
NUM_FRAMES = 3

# TRAINING
BATCH_SIZE = 32 # down from 64 because of frame size
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5
EPOCHS = 100

In [151]:
def prepare_dataset(data_dir, test_size=TEST_SIZE, val_size=VAL_SIZE, random_state=DATASET_SEED):
    video_lengths = []
    videos = os.listdir(data_dir)
    for video_file in videos:
        video_path = os.path.join(data_dir, video_file)
        video_reader = cv2.VideoCapture(video_path)
        frames_count = int(video_reader.get(cv2.CAP_PROP_FRAME_COUNT))

        video_name, _ = os.path.splitext(video_file)
        video_lengths.append((video_path, video_name, frames_count))
        video_reader.release()
        
    train_videos, test_val_videos = train_test_split(video_lengths, test_size=test_size+val_size, random_state=random_state)
    val_videos, test_videos = train_test_split(test_val_videos, test_size=test_size/(test_size+val_size), random_state=random_state)
    
    return train_videos, val_videos, test_videos

In [152]:
def prepare_annotations(lab_dirs):
    all_annotations = {}
    
    for ann_dir in lab_dirs:
        annotations = os.listdir(current_directory)
        for file in annotations:
            file_name, file_extension = os.path.splitext(file)
            if file_extension.lower() == '.csv':
                file_path = os.path.join(current_directory, file)

                file_data = {}
                with open(file_path, mode='r') as reading_file:
                    csv_reader = csv.reader(reading_file)
                    for i, row in enumerate(csv_reader):
                        if i != 0:
                            frame = float(row[0])
                            fixfr = math.floor(frame)
                            frame_num = 3 * (fixfr // 100) + ((fixfr % 100) // 33)
                            file_data[frame_num] = (int(row[1]), int(row[2]))
                            
                # There is only one annotation per file, if this is not true this doesn't work
                all_annotations[file_name] = file_data

    return all_annotations

In [153]:
train_info, val_info, test_info = prepare_dataset(DATA_DIR)
all_annotations = prepare_annotations(LABEL_DIRS)

In [159]:
# Purging all the files that don't have annotations
#i = 0
#for info in test_info:
#    if info[1] not in all_annotations:
#        os.remove(info[0])
#        i += 1
#print(i)

0
